<a href="https://colab.research.google.com/github/Liran-Martfel/Project-Vision-CNN_-in-progress/blob/Code-updates/PROJECT_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "opencv-python-headless<5"

In [ ]:
import zipfile #importing zip actions
from google.colab import drive #library to connect to drive
drive.mount('/content/drive') #connecting with file-path on drive
zipfile_path = '/content/drive/MyDrive/Colab Notebooks/Project - CNN/new_data.zip' #saving the file path under zipfile_path
with zipfile.ZipFile(zipfile_path,'r') as face_exp_data: #from zipfile library, use ZipFile class
  face_exp_data.extractall('/content/data') #extract all data from zipfile
train_data = '/content/data/fer2013plus/fer2013/train'
#since I'm working with drive, i can't use [....] normally, i need to use the path of the files.
test_data = '/content/data/fer2013plus/fer2013/test'

In [3]:
import os
!rm -rf /content/data/fer2013plus/fer2013/train/contempt /content/data/fer2013plus/fer2013/test/contempt
print(sorted(os.listdir('/content/data/fer2013plus/fer2013/train')))

['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']


In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator #importing the library and action to generate images for the cnn so that he can learn the images

# Training data generator with augmentation
train_datagen = ImageDataGenerator (
    rescale = 1./255, # Normalize pixel values to [0,1] — helps speed up training and stabilize gradients
    shear_range = 0.2,  # Apply a slight diagonal transformation (shear) — simulates natural changes in camera angle
    zoom_range = 0.2,  # Apply random zoom-in effect — helps the model recognize objects at different scales
    horizontal_flip = True # Flip images horizontally — helps the model handle symmetry
)
# Testing data generator — only normalization, no augmentation
test_datagen = ImageDataGenerator(rescale=1./255)

In [5]:
training_set = train_datagen.flow_from_directory( #creating the training set which the cnn will train on with that data
    train_data, #the train data from the exctract ziped file
    target_size = (48,48), # the size of the image
    batch_size = 32, # defualt, the number of images that the model sees all at once.
    class_mode = 'categorical',  #useing categorical beacuse every label is becoming a vector, which will help me when i have more then only 2 chcoices, (like emotions)
    color_mode = 'grayscale')

testing_set = test_datagen.flow_from_directory( #creating the testing set which the cnn will test on with that data
    test_data,
    target_size = (48,48),
    batch_size = 32,
    class_mode = 'categorical',
    color_mode = 'grayscale')

Found 28221 images belonging to 7 classes.
Found 7042 images belonging to 7 classes.


In [ ]:
from keras.models import Sequential #importing the empty model (CNN)
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Dropout #importing the 5 types layers of the CNN
cnn = Sequential() #creating the empty CNN model

#first layer, 32 filters of 3X3 weights, 48,48,1 is the original size of the photos & 1 for grayscale
cnn.add(Conv2D (32,(3,3),activation = 'relu', input_shape = (48,48,1))) #define the structure of the layer and add it to the CNN model, using relu as activition as defualt & negatives become 0, also, it's "cheap" and train fast
cnn.add(MaxPooling2D(pool_size = (2,2))) #creating a max_pool with small sqers 2 by 2, and saving the hight number in each sqer

#second layer, 64 filters of 3X3 weights, lose resolution but gain more pattern types.
cnn.add(Conv2D (64,(3,3),activation = 'relu'))
cnn.add(MaxPooling2D(pool_size = (2,2)))

#third layer, 128 filters of 3X3 weights, lose resolution but gain more pattern types.
cnn.add(Conv2D (128,(3,3),activation = 'relu'))
cnn.add(MaxPooling2D(pool_size = (2,2)))
cnn.add(Flatten()) # Flatten, turn the feature maps into one long vector for the layers

#creating the dense
cnn.add(Dense(units = 128,activation= 'relu')) #choose 128 neurons to activate under relu function

#creating a dorpout in order to switch neuron off from the output of the layer before him.
cnn.add(Dropout(0.5)) #0.5 is standart

cnn.add(Dense(units = 7,activation= 'softmax')) #choose 7 neuron beacuse i have 7 emotions to show



In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from keras.callbacks import EarlyStopping

emotion_array = np.unique(training_set.classes)#, return_counts = True) #checking for the least emotion to give them more wieght.
weights = compute_class_weight(class_weight='balanced', classes=emotion_array, y=training_set.classes)
class_weights = dict(enumerate(weights))

cnn.compile(optimizer = 'adam', loss = 'categorical_crossentropy',metrics = ['accuracy'])
# stop training when val_loss stops improving for 5 epochs, and keep the best weights
stopper = EarlyStopping(monitor = 'val_loss',patience = 5, restore_best_weights = True)
cnn.fit (x = training_set, validation_data = testing_set, epochs = 50, callbacks = [stopper],class_weight = class_weights)
cnn.save('/content/drive/MyDrive/Colab Notebooks/Project - CNN/training_model_v3.keras')

   ### Final model: v2
   #v1 (FER-2013): 59% | v2 (FER+): 74.5% | v3 (FER+ with class_weight): 54%
   #The fit above is the v3 experiment. The deployed model is v2, loaded below from the saved file and evaluated on the test set.

Epoch 1/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 176s 197ms/step - accuracy: 0.1333 - loss: 1.9467 - val_accuracy: 0.2070 - val_loss: 1.8905
Epoch 2/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 161s 182ms/step - accuracy: 0.2358 - loss: 1.8677 - val_accuracy: 0.1958 - val_loss: 1.8296
Epoch 3/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 159s 180ms/step - accuracy: 0.2546 - loss: 1.7809 - val_accuracy: 0.2741 - val_loss: 1.7457
Epoch 4/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 160s 181ms/step - accuracy: 0.2926 - loss: 1.7252 - val_accuracy: 0.3780 - val_loss: 1.6193
Epoch 5/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 204s 184ms/step - accuracy: 0.3379 - loss: 1.6629 - val_accuracy: 0.3258 - val_loss: 1.6596
Epoch 6/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 164s 186ms/step - accuracy: 0.3732 - loss: 1.5879 - val_accuracy: 0.4097 - val_loss: 1.4875
Epoch 7/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 165s 187ms/step - accuracy: 0.3854 - loss: 1.5411 - val_accuracy: 0.4609 - val_loss: 1.4252
Epoch 8/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 200s 184ms/step - accuracy: 0.4035 -

In [ ]:
cnn.summary()

In [ ]:
#test rows for the next block of code:
###################################################################################################
#print(img) #<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=447x447 at 0x79C75864E900>
#print(img.size, img.mode)
#print(img.shape, img.dtype, img.min(), img.max())
#print(dir(training_set))
#print (training_set.class_indices) so i can see the emotions order
#prediction_class_name = training_set.class_indices.keys() ,checking for the keys
#print(croped_img.size)
#print(face_detector.empty())

####################################################################################################

In [17]:
from tensorflow.keras.models import load_model
from PIL import Image #Python Image Library
import numpy as np
from google.colab import drive #library to connect to drive
import cv2 # openCV library which have face recegnition trained modelinside

drive.mount('/content/drive') #connecting with file-path on drive
img_test = '/content/drive/MyDrive/Colab Notebooks/Project - CNN/vision-photo.jpg'
cnn_model = load_model('/content/drive/MyDrive/Colab Notebooks/Project - CNN/training_model_v2.keras') #loading V2
prediction_class_name = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise'] #creating the list of emotions so it can work on the website without the training data

def emotions_probability(prediction_class_name,prediction):
  probability = ""
  for i in range(len(prediction_class_name)):
      probability += f"{prediction_class_name[i]}: {prediction[0][i]* 100:.2f}%\n"
  return probability

def organize_img(img):
  img = img.convert('L') # 'L' = grayscale
  img = img.resize((48,48)) # resizing the img to fit the model (48,48)
  img = np.array(img) #transforming the img into an array in order to normalize it
  img = img / 255 #normaliting the img
  img = img.reshape(1,48,48,1)
  return img

def predict(img, cnn_model):
  prediction = cnn_model.predict(img)
  prediction_output = np.argmax(prediction) #taking the place of the highest propability
  full_prediction = prediction_class_name[prediction_output] # position into emotion name
  return prediction, prediction_output, full_prediction

def create_answer(prediction, prediction_output, full_prediction):
  if prediction[0][prediction_output] < 0.6:
    answer = f"Unknown, I'm only {prediction[0][prediction_output] * 100:.2f}% sure that this emotion is: {full_prediction}"
  else:
    answer = f"I'm {prediction[0][prediction_output] * 100:.2f}% sure that this emotion is: {full_prediction}"
  return answer

face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def face_locating (img_test):
  face_img = Image.open(img_test)
  face_img = face_img.convert('L')
  PIL_img = face_img # keep the grayscale PIL image before it becomes an array
  face_img = np.array(face_img)
  return face_img,PIL_img

def face_ditector(face_img,PIL_img):
  faces = face_detector.detectMultiScale(face_img)
  if len (faces) == 0:
    print ("I'm sorry, i couldn't ditect a face to analize")
    return PIL_img
  x = faces[0][0]
  y = faces[0][1]
  w = faces[0][2]
  h = faces[0][3]
  croped_img = PIL_img.crop((x, y, x + w, y + h))  # crop to the face box: (left, top, right, bottom); right = x+w, bottom = y+h
  return croped_img

# How common each emotion was in the FER+ training set (same order as prediction_class_name)
train_counts = np.array([2466, 191, 652, 7528, 10308, 3514, 3562])
train_prior = train_counts / train_counts.sum()
def balance_prediction(prediction, alpha=0.5):
  # remove part of the model's habit of guessing the common emotions (neutral, happy)
  balanced = prediction / train_prior ** alpha
  balanced = balanced / balanced.sum()                   # back to 100%
  prediction_output = np.argmax(balanced)                # position of the highest probability
  full_prediction = prediction_class_name[prediction_output]
  return balanced, prediction_output, full_prediction

face_img,PIL_img = face_locating(img_test)
croped_img = face_ditector(face_img,PIL_img)
img = organize_img(croped_img)
prediction, prediction_output, full_prediction = predict(img, cnn_model)
prediction, prediction_output, full_prediction = balance_prediction(prediction)
answer = create_answer(prediction, prediction_output, full_prediction)
emotions = emotions_probability(prediction_class_name,prediction)
print(emotions)
print(answer)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
angry: 0.02%
disgust: 0.00%
fear: 3.42%
happy: 0.02%
neutral: 2.17%
sad: 0.18%
surprise: 94.19%

I'm 94.19% sure that this emotion is: surprise


In [18]:
# Evaluation of the final model (v2) on the test set, WITHOUT prior correction.
# shuffle=False keeps the images in order, so the predictions match the true labels.
# Result: accuracy 74.5%, but the model leans toward neutral/happy (disgust 0%, fear 22%, sad 37% recall).

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

evaluation_set = test_datagen.flow_from_directory(
    test_data, target_size=(48, 48), batch_size=32,
    class_mode='categorical', color_mode='grayscale', shuffle=False)

y_true = evaluation_set.classes
y_pred = np.argmax(cnn_model.predict(evaluation_set), axis=1)
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=prediction_class_name))

Found 7042 images belonging to 7 classes.
221/221 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step
[[ 351    0    0   56  162   32   37]
 [  14    0    1    7   23   10    2]
 [   5    0   37   19   26    7   73]
 [  37    0    0 1593  124   34   39]
 [  27    0    1  118 2239  152   60]
 [  35    0    9   77  405  317   13]
 [  31    0    7   37  101   13  711]]
              precision    recall  f1-score   support

       angry       0.70      0.55      0.62       638
     disgust       0.00      0.00      0.00        57
        fear       0.67      0.22      0.33       167
       happy       0.84      0.87      0.85      1827
     neutral       0.73      0.86      0.79      2597
         sad       0.56      0.37      0.45       856
    surprise       0.76      0.79      0.77       900

    accuracy                           0.75      7042
   macro avg       0.61      0.52      0.54      7042
weighted avg       0.73      0.75      0.73      7042



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [19]:
# Same model and same test set, WITH prior correction (alpha = 0.5), as used in the app.
# Each probability is divided by how common that emotion was in training (to the power 0.5), then renormalized,
# so rare emotions (disgust, fear, sad) get a fairer chance.
# Expected: accuracy drops slightly (~73.8%), macro recall rises (52.4% -> ~60%).

train_counts = np.array([2466, 191, 652, 7528, 10308, 3514, 3562])
train_prior = train_counts / train_counts.sum()

test_pred = cnn_model.predict(evaluation_set)
balanced = test_pred / train_prior ** 0.5
y_pred_balanced = balanced.argmax(axis=1)

print(confusion_matrix(y_true, y_pred_balanced))
print(classification_report(y_true, y_pred_balanced, target_names=prediction_class_name))

221/221 ━━━━━━━━━━━━━━━━━━━━ 11s 48ms/step
[[ 409    1    7   42  104   45   30]
 [  18   12    2    5   11    7    2]
 [   9    0   74   11   17    8   48]
 [  61    2   14 1561   87   63   39]
 [  84    1   31  119 1993  296   73]
 [  59    0   28   64  266  428   11]
 [  41    0   31   26   66   19  717]]
              precision    recall  f1-score   support

       angry       0.60      0.64      0.62       638
     disgust       0.75      0.21      0.33        57
        fear       0.40      0.44      0.42       167
       happy       0.85      0.85      0.85      1827
     neutral       0.78      0.77      0.78      2597
         sad       0.49      0.50      0.50       856
    surprise       0.78      0.80      0.79       900

    accuracy                           0.74      7042
   macro avg       0.67      0.60      0.61      7042
weighted avg       0.74      0.74      0.74      7042

